# 🍺 Atividade Prática: Medalhão (Bronze → Prata → Ouro) com Spark e Delta Lake

**Disciplina:** Arquitetura de Coleta e Armazenamento de Dados (Object Storage + Spark)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wuldson-franco/arqu_coleta_dados_pos_unipe/blob/main/Atividade_Pratica_Spark/atividade_medalhao_breweries.ipynb) -> *CLIQUE AQUI PARA ABRIR O ARQUIVO NO SEU GOOGLE COLAB*

| | |
|---|---|
| **Nome do aluno** | _escreva aqui_ |
| **Data** | _escreva aqui_ |

---

## Como fazer esta atividade

1. **Não edite este arquivo diretamente.** Vá em **Arquivo → Salvar uma cópia no Drive** e trabalhe na sua cópia.
2. Rode as células **em ordem**, de cima para baixo. Se der erro estranho, use **Ambiente de execução → Reiniciar sessão** e rode tudo de novo.
3. As perguntas marcadas com ✏️ são **abertas**: responda com as suas palavras, na célula de texto logo abaixo da pergunta.
4. Ao terminar, rode a célula final de **autoverificação** e envie o link da sua cópia diretamente do seu repositorio no Github (com permissão de leitura).

## O que vamos construir

```
 arquivo .parquet  ──►  BRONZE (Delta)  ──►  PRATA (Delta)  ──►  OURO (Delta)
   (dado cru)           dado como chegou     limpo e corrigido    agregado p/ o negócio
```

**Pergunta de negócio:** _"Quais tipos de cervejaria existem por cidade, província e país?"_

> 💡 **Sobre o uso de IA:** você pode usar IA como apoio (Claude, ChatGPT, Gemini, Copilot…). Mas **você** responde pelo resultado. Duas perguntas no final pedem que você reflita sobre esse uso, então guarde o que perguntou e o que você precisou corrigir.

## 0. Preparando o ambiente

Vamos instalar o Java, o PySpark e o Delta Lake. As versões precisam combinar entre si (PySpark 3.5.1 ↔ Delta 3.1.0). **Esta célula demora 1 a 2 minutos.**

In [ ]:
%%bash

# Instalar Java
apt-get update --fix-missing
apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Instalar PySpark e Delta Lake (versões compatíveis para evitar conflito de dependências)
pip uninstall -y delta-spark
pip install -q pyspark==3.5.1
pip install -q delta-spark==3.1.0

In [ ]:
import os, logging, requests

# Apontamos o Spark para o Java que acabamos de instalar
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName("Atividade_Medalhao_Breweries")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.repl.eagerEval.enabled", True)
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark", spark.version, "pronto ✅")

# Nosso "data lake" de brincadeira: pastas no disco da VM do Colab
base_path = "/content/datalake"
for camada in ["bronze", "silver", "gold"]:
    os.makedirs(os.path.join(base_path, camada), exist_ok=True)
print("Pastas criadas em", base_path)

---
## 1. 🥉 Camada Bronze

Vamos baixar o arquivo cru e guardá-lo na Bronze **sem alterar nada**, só convertendo para o formato Delta.

In [ ]:
parquet_url  = "https://github.com/wuldson-franco/breweries_case/raw/main/analises/bs_bronze.parquet"
parquet_path = os.path.join(base_path, "bronze", "bs_bronze.parquet")
delta_bronze = os.path.join(base_path, "bronze", "bs_bronze_delta")

# Baixar o arquivo Parquet (só se ainda não existir)
if not os.path.exists(parquet_path):
    response = requests.get(parquet_url)
    response.raise_for_status()
    with open(parquet_path, "wb") as f:
        f.write(response.content)

df = spark.read.parquet(parquet_path)

# mode("overwrite") permite rodar a célula mais de uma vez sem erro
df.write.format("delta").mode("overwrite").save(delta_bronze)

print("Linhas na Bronze:", spark.read.format("delta").load(delta_bronze).count())

Agora vamos **olhar por dentro** do que foi gravado. Em um bucket de object storage, cada arquivo seria um _objeto_ e o caminho completo seria a sua _chave_.

In [ ]:
# Lista tudo o que existe dentro da Bronze Delta (caminho relativo e tamanho)
for raiz, _, arquivos in os.walk(delta_bronze):
    for nome in sorted(arquivos):
        caminho = os.path.join(raiz, nome)
        print(f"{os.path.relpath(caminho, base_path):<85} {os.path.getsize(caminho):>10,} bytes")

### ✏️ Q1. Olhando por dentro da Bronze
Observe a listagem acima. **(a)** Que tipos de arquivo você vê e para que serve cada um? **(b)** Se essa pasta estivesse em um bucket S3/MinIO, o que seriam a _chave_ e o _objeto_ de um desses arquivos?

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

### ✏️ Q2. Por que Bronze e por que Delta?
**(a)** Por que a Bronze deve guardar o dado **como ele chegou**, sem corrigir nada? **(b)** Qual problema o Delta resolve que um monte de arquivos Parquet soltos no storage não resolve?

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

---
## 2. 🥈 Camada Prata

Aqui o dado é inspecionado, limpo e corrigido. Resolva os passos na ordem.

**2.1** Exiba o _schema_ das colunas com `printSchema`.

In [ ]:
# SEU CÓDIGO AQUI

**2.2** Faça um `select` do país para verificar **quais países constam na nossa base**. Mostre também **quantas cervejarias** há em cada um.

In [ ]:
# SEU CÓDIGO AQUI

**2.3** Descubra **quais cervejarias estão com o endereço faltante**.

> Dica: "faltante" pode ser `null` **e também** texto vazio (`""`) ou só espaços. Confira os dois casos. Qual é a coluna de endereço? Veja no `printSchema`.

In [ ]:
# SEU CÓDIGO AQUI

**2.4** Pesquise na internet **qual é o endereço real** dessas cervejarias e inclua-os nos campos faltantes usando `when` / `otherwise`.

Crie também uma coluna `address_fonte` dizendo **de onde veio o dado** (por exemplo: `"original"` ou `"pesquisa manual"`). Guarde esta ideia: ela será cobrada na pergunta Q3.

> Estrutura do `when` (os valores abaixo são só um **exemplo**, troque pelos seus):
> ```python
> df_silver = df.withColumn(
>     "address_1",
>     F.when(F.col("id") == "ID_DA_CERVEJARIA", "Endereço encontrado").otherwise(F.col("address_1"))
> )
> ```

In [ ]:
# SEU CÓDIGO AQUI
# df_silver = ...

**2.5** Verificações de qualidade. Responda com código:
- Existem `id` **duplicados**?
- Existem nulos em `brewery_type`, `city` ou `country`?
- Depois da sua correção, ainda sobra algum endereço faltante?

In [ ]:
# SEU CÓDIGO AQUI

**2.6** Salve a Prata em **Delta**, em `silver/breweries_silver`, e depois leia de volta para conferir a contagem.

In [ ]:
silver_path = os.path.join(base_path, "silver", "breweries_silver")

# SEU CÓDIGO AQUI (gravar com mode("overwrite"))

# SEU CÓDIGO AQUI (ler de volta e dar count)

### ✏️ Q3. Qualidade e governança
**(a)** Você corrigiu os endereços na mão, com `when/otherwise`. Isso funcionaria para 5 milhões de registros com problemas? Que alternativa você proporia? **(b)** Para que serve a coluna `address_fonte` que você criou? Relacione com os conceitos de **linhagem** e **confiança no dado**.

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

---
## 3. 🥇 Camada Ouro

Vamos responder à pergunta de negócio: **"Quais tipos de cervejaria por cidade, província e país?"**

Abaixo está a função de agregação, escrita em **pandas**. Ela não roda direto em um DataFrame do Spark. Sua tarefa é **convertê-la para Spark**.

In [ ]:
# ATENÇÃO: esta função é pandas. Não rode com o DataFrame do Spark.
def data_aggregation(df):
    try:
        # Agrupando por tipo de cervejaria e localização
        aggregated_df = df.groupby(['brewery_type', 'city', 'state_province', 'country']).size().reset_index(name='brewery_count')
        return aggregated_df
    except Exception as e:
        logging.error(f"Erro ao realizar a agregação de dados: {e}")
        raise

In [ ]:
# SEU CÓDIGO AQUI: escreva a versão em Spark, usando groupBy + agg (ou count)
def data_aggregation_spark(df):
    ...

Agora aplique a função na **Prata**, salve o resultado em **Delta** em `gold/breweries_por_local` e mostre as 10 linhas com maior `brewery_count`.

In [ ]:
gold_path = os.path.join(base_path, "gold", "breweries_por_local")

# SEU CÓDIGO AQUI

### ✏️ Q4. Para quem é a Ouro?
**(a)** Por que não deixar o dashboard ler a Prata e fazer essa agregação na hora? **(b)** Cite dois consumidores possíveis dessa tabela Ouro e o que cada um faria com ela.

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

---
## 4. 🧊 Bônus Delta: viagem no tempo

O Delta guarda um **histórico de versões** no `_delta_log`. Rode a célula abaixo (ela já está pronta) e depois responda.

In [ ]:
from delta.tables import DeltaTable

# Reescreve a Prata uma segunda vez (gera uma nova versão)
spark.read.format("delta").load(silver_path).write.format("delta").mode("overwrite").save(silver_path)

DeltaTable.forPath(spark, silver_path).history().select("version", "timestamp", "operation").show(truncate=False)

# Ler a versão 0, como estava no início
spark.read.format("delta").option("versionAsOf", 0).load(silver_path).count()

### ✏️ Q5. Time travel
Em que situação real de trabalho você usaria `versionAsOf`? Dê um exemplo e explique **como isso é possível** em um storage onde os objetos não são editados no lugar.

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

---
## 5. 🪣 Arquitetura: Object Storage

Estas perguntas não têm código. Elas medem se você entendeu **onde** e **como** os dados vivem.

### ✏️ Q6. Bucket, objeto e "pasta"
Explique a diferença entre **object storage** (S3, MinIO, GCS) e um **sistema de arquivos** comum. Por que se diz que em object storage "pasta não existe de verdade"? O que isso muda na hora de **renomear** ou **listar** dados?

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

### ✏️ Q7. Storage separado do processamento
Spark e object storage são sistemas separados, ligados pela rede. **Cite uma vantagem e um custo** (ou risco) dessa separação. Se quiser, comente também o problema dos "arquivos pequenos".

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

### ✏️ Q8. Do Colab para o S3
Nesta atividade gravamos em `/content/datalake`, no disco da máquina do Colab. **(a)** O que acontece com esses dados quando a sessão do Colab é encerrada? **(b)** Se o destino fosse um bucket (`s3a://meu-bucket/bronze/...`), o que mudaria no código e na configuração? **(c)** Que cuidados de **segurança** você teria?

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

### ✏️ Q9. Projetando para escala
Imagine que a tabela Ouro passou a ter **500 milhões de linhas por ano**. Como você organizaria os arquivos no storage (formato, particionamento, tamanho dos arquivos)? Justifique pensando em **quem consulta** e **com que frequência**.

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

---
## 6. 🤖 Uso de IA

### ✏️ IA-1. Você e o assistente de IA
Descreva **como você usou IA** nesta atividade (ou como usaria): para pesquisar os endereços, converter a função pandas para Spark, depurar erros? **(a)** Cole o prompt principal que você usou. **(b)** Como você **conferiu** que a resposta estava certa? **(c)** Teve algo que a IA errou ou inventou? Se não usou IA, explique como você usaria e quais riscos checaria (pense em endereços inventados e em código que "parece certo" mas dá resultado diferente).

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

### ✏️ IA-2. A qualidade do dado e a IA
Hoje as camadas Prata e Ouro costumam alimentar dashboards, modelos de ML e aplicações de IA generativa (como assistentes que respondem sobre os dados). Usando **exemplos desta base** (endereços faltantes, duplicados, tipos de cervejaria), explique como a **qualidade do dado** nas camadas anteriores afeta a **confiabilidade** do que a IA responde. O que você colocaria no pipeline para evitar respostas erradas?

**✏️ Sua resposta:**

> _Dê dois cliques nesta célula e escreva com as suas palavras. Não copie definição pronta: explique como se estivesse ensinando um colega._

---
## 7. 🏁 Bônus opcional: gráfico

Crie um gráfico (Spark + matplotlib, ou pandas) com os dados da Ouro. Sugestões: top 10 países por número de cervejarias, ou distribuição por `brewery_type`.

> Dica: use `.toPandas()` **só depois de agregar**, para trazer poucas linhas ao driver.

In [ ]:
import matplotlib.pyplot as plt

# SEU CÓDIGO AQUI

---
## ✅ Autoverificação

Rode esta célula antes de entregar. Ela confere **se os arquivos existem** e mostra as contagens. Ela não diz se o seu resultado está certo.

In [ ]:
checagens = {
    "Bronze": os.path.join(base_path, "bronze", "bs_bronze_delta"),
    "Prata ": os.path.join(base_path, "silver", "breweries_silver"),
    "Ouro  ": os.path.join(base_path, "gold",   "breweries_por_local"),
}
for nome, caminho in checagens.items():
    try:
        n = spark.read.format("delta").load(caminho).count()
        print(f"✅ {nome}: {n:,} linhas  ({os.path.relpath(caminho, base_path)})")
    except Exception as e:
        print(f"❌ {nome}: não encontrada em {os.path.relpath(caminho, base_path)}")

print("\nLembrete: responda TODAS as perguntas ✏️ (Q1 a Q9, IA-1 e IA-2) antes de enviar.")